In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
import os

In [2]:
materials_path = "../data/processed/materials.csv"
products_path = "../data/processed/products.csv"

materials = pd.read_csv(materials_path)
products = pd.read_csv(products_path)

print("=== MATERIALS INFO ===")
print(materials.info())
print("\nMissing values (materials):")
print(materials.isna().sum())

print("\n=== PRODUCTS INFO ===")
print(products.info())
print("\nMissing values (products):")
print(products.isna().sum())

=== MATERIALS INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   material_id               404 non-null    object 
 1   material_type             404 non-null    object 
 2   strength_mpa              404 non-null    int64  
 3   weight_capacity           404 non-null    int64  
 4   biodegradability_percent  404 non-null    int64  
 5   co2_emission_score        403 non-null    float64
 6   recyclability_percent     404 non-null    int64  
 7   cost_per_kg               404 non-null    float64
 8   industry_use_case         404 non-null    object 
dtypes: float64(2), int64(4), object(3)
memory usage: 28.5+ KB
None

Missing values (materials):
material_id                 0
material_type               0
strength_mpa                0
weight_capacity             0
biodegradability_percent    0
co2_emission_score   

In [4]:
# Materials dataset
materials_num_cols = [
    "strength_mpa",
    "weight_capacity",
    "biodegradability_percent",
    "co2_emission_score",
    "recyclability_percent",
    "cost_per_kg",
]

materials_cat_cols = [
    "material_type",
    "industry_use_case",
    # "material_id"  # usually ID, normally not imputed
]

# Products dataset
products_num_cols = [
    "product_weight",
    "fragility_index",
]

products_cat_cols = [
    "product_name",
    "category",
    "shipping_type",
    # "product_id"  # usually ID, normally not imputed
]

In [5]:
materials_num_cols = [c for c in materials_num_cols if c in materials.columns]
materials_cat_cols = [c for c in materials_cat_cols if c in materials.columns]

products_num_cols = [c for c in products_num_cols if c in products.columns]
products_cat_cols = [c for c in products_cat_cols if c in products.columns]

print("\nNumeric columns (materials):", materials_num_cols)
print("Categorical columns (materials):", materials_cat_cols)
print("\nNumeric columns (products):", products_num_cols)
print("Categorical columns (products):", products_cat_cols)


Numeric columns (materials): ['strength_mpa', 'weight_capacity', 'biodegradability_percent', 'co2_emission_score', 'recyclability_percent', 'cost_per_kg']
Categorical columns (materials): ['material_type', 'industry_use_case']

Numeric columns (products): ['product_weight', 'fragility_index']
Categorical columns (products): ['product_name', 'category', 'shipping_type']


In [6]:
# Numeric: median
num_imputer = SimpleImputer(strategy="median")

# Categorical: most frequent
cat_imputer = SimpleImputer(strategy="most_frequent")

In [7]:
if materials_num_cols:
    materials[materials_num_cols] = num_imputer.fit_transform(materials[materials_num_cols])

if materials_cat_cols:
    materials[materials_cat_cols] = cat_imputer.fit_transform(materials[materials_cat_cols])

print("\nAfter imputation (materials) missing values:")
print(materials.isna().sum())


After imputation (materials) missing values:
material_id                 0
material_type               0
strength_mpa                0
weight_capacity             0
biodegradability_percent    0
co2_emission_score          0
recyclability_percent       0
cost_per_kg                 0
industry_use_case           0
dtype: int64


In [8]:

if products_num_cols:
    products[products_num_cols] = num_imputer.fit_transform(products[products_num_cols])

if products_cat_cols:
    products[products_cat_cols] = cat_imputer.fit_transform(products[products_cat_cols])

print("\nAfter imputation (products) missing values:")
print(products.isna().sum())



After imputation (products) missing values:
product_id         0
product_name       0
category           0
product_weight     0
fragility_index    0
shipping_type      0
dtype: int64


In [9]:
os.makedirs("../data/processed", exist_ok=True)

clean_mat_path = "../data/processed/cleaned_materials.csv"
clean_prod_path = "../data/processed/cleaned_products.csv"

materials.to_csv(clean_mat_path, index=False)
products.to_csv(clean_prod_path, index=False)

print(f"\n[SAVED] Cleaned materials -> {clean_mat_path}")
print(f"[SAVED] Cleaned products  -> {clean_prod_path}")


[SAVED] Cleaned materials -> ../data/processed/cleaned_materials.csv
[SAVED] Cleaned products  -> ../data/processed/cleaned_products.csv


In [10]:
missing_table = pd.DataFrame({
    "materials_missing": materials.isna().sum(),
    "products_missing": products.isna().sum()
})

missing_table_path = "../notebooks/missing_value_table.csv"
os.makedirs("../notebooks", exist_ok=True)
missing_table.to_csv(missing_table_path)

print(f"\n[SAVED] Missing value table -> {missing_table_path}")


[SAVED] Missing value table -> ../notebooks/missing_value_table.csv


In [23]:
# TASK 2: ENCODING + NORMALIZATION
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import joblib
import os
clean_mat_path = "../data/processed/cleaned_materials.csv"
clean_prod_path = "../data/processed/cleaned_products.csv"

model_ready_dir = "../data/model_ready"
encoders_dir = "../models/encoders"
scalers_dir = "../models/scalers"

os.makedirs(model_ready_dir, exist_ok=True)
os.makedirs(encoders_dir, exist_ok=True)
os.makedirs(scalers_dir, exist_ok=True)

materials = pd.read_csv(clean_mat_path)
products = pd.read_csv(clean_prod_path)

print("Cleaned materials shape:", materials.shape)
print("Cleaned products shape :", products.shape)

print("\n=== CLEANED MATERIALS (head) ===")
print(materials.head())

print("\n=== CLEANED PRODUCTS (head) ===")
print(products.head())

mat_id_col = ["material_id"]  # keep as ID, not encode
mat_cat_cols = ["material_type", "industry_use_case"]
mat_num_cols = [
    "strength_mpa",
    "weight_capacity",
    "biodegradability_percent",
    "co2_emission_score",
    "recyclability_percent",
    "cost_per_kg",
]

# only keep columns that actually exist
mat_cat_cols = [c for c in mat_cat_cols if c in materials.columns]
mat_num_cols = [c for c in mat_num_cols if c in materials.columns]

print("\n[Materials] Categorical (to encode):", mat_cat_cols)
print("[Materials] Numeric (to scale):", mat_num_cols)

# ---- PRODUCTS ----
prod_id_col = ["product_id"]  # keep as ID
prod_cat_cols = ["product_name", "category", "shipping_type"]
prod_num_cols = ["product_weight", "fragility_index"]

prod_cat_cols = [c for c in prod_cat_cols if c in products.columns]
prod_num_cols = [c for c in prod_num_cols if c in products.columns]

print("\n[Products] Categorical (to encode):", prod_cat_cols)
print("[Products] Numeric (to scale):", prod_num_cols)

if "fragility_index" in prod_num_cols:
    products["fragility_index"] = pd.to_numeric(products["fragility_index"], errors="coerce")

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

mat_ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
mat_cat_encoded = mat_ohe.fit_transform(materials[mat_cat_cols])
mat_ohe_cols = mat_ohe.get_feature_names_out(mat_cat_cols)
mat_cat_df = pd.DataFrame(mat_cat_encoded, columns=mat_ohe_cols, index=materials.index)

mat_scaler = MinMaxScaler()
mat_num_scaled = mat_scaler.fit_transform(materials[mat_num_cols])
mat_num_df = pd.DataFrame(mat_num_scaled, columns=mat_num_cols, index=materials.index)

materials_final = pd.concat(
    [
        materials[mat_id_col].reset_index(drop=True),
        mat_num_df.reset_index(drop=True),
        mat_cat_df.reset_index(drop=True),
    ],
    axis=1,
)

materials_encoded_path = os.path.join(model_ready_dir, "materials_final_encoded.csv")
materials_final.to_csv(materials_encoded_path, index=False)
print(f"\n[SAVED] Encoded materials -> {materials_encoded_path}")

# 4.5 Save encoders & scalers
joblib.dump(mat_ohe, os.path.join(encoders_dir, "materials_ohe.pkl"))
joblib.dump(mat_scaler, os.path.join(scalers_dir, "materials_scaler.pkl"))
print("[SAVED] materials_ohe.pkl and materials_scaler.pkl")

prod_ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
prod_cat_encoded = prod_ohe.fit_transform(products[prod_cat_cols])
prod_ohe_cols = prod_ohe.get_feature_names_out(prod_cat_cols)
prod_cat_df = pd.DataFrame(prod_cat_encoded, columns=prod_ohe_cols, index=products.index)

# 5.2 Scale numeric columns
prod_scaler = MinMaxScaler()
prod_num_scaled = prod_scaler.fit_transform(products[prod_num_cols])
prod_num_df = pd.DataFrame(prod_num_scaled, columns=prod_num_cols, index=products.index)

# 5.3 Combine: ID + scaled numeric + encoded categoricals
products_final = pd.concat(
    [
        products[prod_id_col].reset_index(drop=True),
        prod_num_df.reset_index(drop=True),
        prod_cat_df.reset_index(drop=True),
    ],
    axis=1,
)

# 5.4 Save encoded products dataset
products_encoded_path = os.path.join(model_ready_dir, "products_final_encoded.csv")
products_final.to_csv(products_encoded_path, index=False)
print(f"[SAVED] Encoded products -> {products_encoded_path}")

# 5.5 Save encoders & scalers
joblib.dump(prod_ohe, os.path.join(encoders_dir, "products_ohe.pkl"))
joblib.dump(prod_scaler, os.path.join(scalers_dir, "products_scaler.pkl"))
print("[SAVED] products_ohe.pkl and products_scaler.pkl")


print("\n=== Encoded materials sample ===")
print(materials_final.head())

print("\n=== Encoded products sample ===")
print(products_final.head())

print("\n✅ Task 2 completed for both datasets.")


Cleaned materials shape: (404, 9)
Cleaned products shape : (404, 6)

=== CLEANED MATERIALS (head) ===
  material_id    material_type  strength_mpa  weight_capacity  \
0    MAT_0001        Cardboard          32.0             50.0   
1    MAT_0002  Paper/Bio-Based          20.0             30.0   
2    MAT_0003            Steel         250.0            500.0   
3    MAT_0004  Paper/Bio-Based          20.0             30.0   
4    MAT_0005  Paper/Bio-Based          20.0             30.0   

   biodegradability_percent  co2_emission_score  recyclability_percent  \
0                      95.0                0.78                   98.0   
1                      98.0                0.52                  100.0   
2                       0.0                3.67                   87.0   
3                      98.0                0.48                  100.0   
4                      98.0                0.41                  100.0   

   cost_per_kg                                  industry_use_c